# 토큰 예산으로 대화 컨텍스트 자르기

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

과거의 `ConversationTokenBufferMemory` 대신 현재는 `trim_messages()`로 모델 호출 직전
메시지를 자르고, 원본 대화 상태는 checkpointer에 보존하는 패턴을 사용합니다.

`trim_messages()`는 메시지 역할 순서와 시스템 메시지를 보존하는 옵션을 제공하므로,
문자열을 임의로 잘라내는 것보다 안전합니다.


In [1]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv


In [2]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.6-luna")
model = init_chat_model(MODEL_ID)


## API 호출 없이 trimming 동작 확인

`token_counter="approximate"`는 빠르고 공급자에 독립적인 근사치입니다. 청구량처럼
정확한 값이 필요할 때는 사용하는 모델을 token counter로 전달할 수 있습니다.


In [3]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage, trim_messages
from langchain_core.messages.utils import count_tokens_approximately

MAX_TOKENS = 80
SYSTEM_PROMPT = "당신은 공작 기계 설치를 돕는 기술 지원 상담원입니다."

sample_messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="최근에 XG-200 공작 기계를 구매했습니다. 설치를 도와주세요."),
    AIMessage(content="먼저 설치 장소의 220V 전원과 접지 상태를 확인해 주세요."),
    HumanMessage(content="전원과 접지를 확인했습니다. 다음은 무엇인가요?"),
    AIMessage(content="평평하고 안정된 바닥에 기계를 두고 수평계를 사용해 수평을 맞추세요."),
    HumanMessage(content="수평을 맞췄습니다. 케이블은 어떻게 연결하나요?"),
    AIMessage(content="전원을 차단한 상태에서 매뉴얼 5쪽의 배선도 순서대로 연결하세요."),
    HumanMessage(content="연결을 마쳤습니다. 이제 무엇을 확인해야 하나요?"),
    AIMessage(content="보호 덮개와 비상 정지 버튼을 확인한 뒤 초기 구동 테스트를 진행하세요."),
]

trimmed = trim_messages(
    sample_messages,
    max_tokens=MAX_TOKENS,
    token_counter="approximate",
    strategy="last",
    start_on="human",
    include_system=True,
    allow_partial=False,
)

print("원본 근사 토큰:", count_tokens_approximately(sample_messages))
print("정리 후 근사 토큰:", count_tokens_approximately(trimmed))
print()
for message in trimmed:
    print(f"[{message.type}] {message.content}")


원본 근사 토큰: 118
정리 후 근사 토큰: 65

[system] 당신은 공작 기계 설치를 돕는 기술 지원 상담원입니다.
[human] 수평을 맞췄습니다. 케이블은 어떻게 연결하나요?
[ai] 전원을 차단한 상태에서 매뉴얼 5쪽의 배선도 순서대로 연결하세요.
[human] 연결을 마쳤습니다. 이제 무엇을 확인해야 하나요?
[ai] 보호 덮개와 비상 정지 버튼을 확인한 뒤 초기 구동 테스트를 진행하세요.


## LangGraph 호출 직전에 같은 정책 적용하기

그래프 상태에는 전체 메시지를 저장하고, `call_model` 노드 안에서만 토큰 예산에 맞춘
목록을 사용합니다.


In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph


def trim_for_model(messages):
    return trim_messages(
        [SystemMessage(content=SYSTEM_PROMPT), *messages],
        max_tokens=MAX_TOKENS,
        token_counter="approximate",
        strategy="last",
        start_on="human",
        include_system=True,
        allow_partial=False,
    )


def call_model(state: MessagesState):
    response = model.invoke(trim_for_model(state["messages"]))
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")
builder.add_edge("model", END)
token_chat = builder.compile(checkpointer=InMemorySaver())


In [5]:
config = {"configurable": {"thread_id": "token-demo"}}

# 기존 예시 대화와 새 질문을 첫 checkpoint 입력으로 넣습니다.
result = token_chat.invoke(
    {
        "messages": [
            *sample_messages[1:],  # 시스템 메시지는 노드에서 별도로 추가
            HumanMessage(content="마지막으로 해야 할 점검을 한 문장으로 알려주세요."),
        ]
    },
    config=config,
)
print(result["messages"][-1].content)


모든 보호커버·접지·윤활 상태와 비상정지 기능을 확인한 후, 저속 무부하로 시운전하여 이상 진동·소음·누유가 없는지 점검하세요.


In [6]:
full_history = token_chat.get_state(config).values["messages"]
context_used_for_last_call = trim_for_model(full_history[:-1])

print("저장된 전체 메시지 수:", len(full_history))
print("마지막 모델 입력 메시지 수:", len(context_used_for_last_call))


저장된 전체 메시지 수: 10
마지막 모델 입력 메시지 수: 6
